# Combined ML + Strategy Predictions

Ensemble approach combining LSTM predictions and scalping strategy signals for improved trading decisions.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

print("="*80)
print("COMBINED ML + STRATEGY BACKTESTING")
print("="*80)
print(f"Testing Period: {TEST_START} to {TEST_END}")
print("="*80)

COMBINED ML + STRATEGY BACKTESTING
Testing Period: 2024-01-01 to 2024-12-31


## Define Feature Engineering and Scalping Strategy

Replicate feature engineering and strategy logic from earlier notebooks.

In [6]:
# ======================================================
# SCALPING STRATEGY SIGNALS (Rule-Based)
# ======================================================
def add_scalping_signals(data):
    df = data.copy()

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))

    # Moving averages
    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    # MACD
    ema_12 = df["Close"].ewm(span=12).mean()
    ema_26 = df["Close"].ewm(span=26).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9).mean()
    macd_hist = macd - macd_signal

    # BUY conditions
    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = rsi < 40
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    buy_signal = (
        (buy_uptrend & buy_rsi) |
        (buy_uptrend & buy_macd) |
        (buy_uptrend & buy_strength)
    )

    # SELL conditions
    sell_downtrend = (df["Close"] < sma_20) | (sma_20 < sma_50)
    sell_rsi = rsi > 60
    sell_macd = (macd < 0) & (macd_hist < 0)

    sell_signal = (
        (sell_downtrend & sell_rsi) |
        (sell_downtrend & sell_macd)
    )

    # Final signal
    signal = pd.Series(0, index=df.index)
    signal[buy_signal] = 1
    signal[sell_signal] = -1
    signal[(buy_signal) & (sell_signal)] = 1

    df["strategy_signal"] = signal
    return df


# ======================================================
# FEATURE ENGINEERING (ML + Trading Aligned)
# ======================================================
def add_basic_features(data, horizon=3, cost=0.0003):
    df = data.copy()

    # Returns
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # Trend
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / sma_10
    df["trend_20"] = (df["Close"] - sma_20) / sma_20
    df["trend_diff"] = (sma_10 - sma_20) / sma_20

    # Price action
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    df["body_pct"] = (df["Close"] - df["Open"]) / df["Close"]
    df["body_abs"] = df["body_pct"].abs()

    # Volatility regime
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / df["volatility_10"].rolling(50).mean()
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # RSI (0–1)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # Volume
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # Target: forward return beyond cost
    future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
    df["target"] = (future_return > cost).astype(int)

    df.dropna(inplace=True)
    return df






## Load Data and Generate Both ML + Strategy Predictions

For first ticker, compare:
1. ML predictions (LSTM only)
2. Strategy signals (technical rules only)
3. Combined predictions (voting ensemble)

In [7]:
ticker = DEFAULT_TICKERS[0]
print(f"\n{'='*80}")
print(f"ANALYZING {ticker}")
print(f"{'='*80}")

# Load and prepare data
raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)
train_data, test_data = split_data_by_date(cleaned_data)

print(f"Train data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

# Feature engineering for ML
# ------------------------------------------------------
# Apply strategy FIRST (raw data)
# ------------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
test_with_signals  = add_scalping_signals(test_data)

# ------------------------------------------------------
# Then apply feature engineering (keeps alignment)
# ------------------------------------------------------
train_with_features = add_basic_features(train_with_signals)
test_with_features  = add_basic_features(test_with_signals)

print(f"Train with features: {train_with_features.shape}")
print(f"Test with features:  {test_with_features.shape}")



ANALYZING NIFTY BANK
2025-12-24 18:31:04 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-24 18:31:05 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-24 18:31:05 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-24 18:31:05 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-24 18:31:05 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-24 18:31:05 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-24 18:31:05 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-24 18:31:05 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 0

In [8]:
# ======================================================
# STEP 1: ML PREDICTIONS (XGBoost - IMPROVED)
# ======================================================
print("\n" + "="*80)
print("STEP 1: XGBoost ML MODEL PREDICTIONS (IMPROVED)")
print("="*80)

# ------------------------------------------------------
# Feature selection
# ------------------------------------------------------
feature_cols = [
    col for col in train_with_features.columns
    if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume']
]

X_train = train_with_features[feature_cols]
y_train = train_with_features['target']

X_test = test_with_features[feature_cols]
y_test = test_with_features['target']

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

# ------------------------------------------------------
# Time-based validation split (NO leakage)
# ------------------------------------------------------
val_size = int(0.8 * len(X_train))
X_tr, X_val = X_train.iloc[:val_size], X_train.iloc[val_size:]
y_tr, y_val = y_train.iloc[:val_size], y_train.iloc[val_size:]

# ------------------------------------------------------
# XGBoost model (tuned for noisy financial data)
# ------------------------------------------------------
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=20,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

print("\nTraining XGBoost model...")

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)



# ------------------------------------------------------
# Probabilities
# ------------------------------------------------------
y_val_prob = xgb_model.predict_proba(X_val)[:, 1]
y_test_prob = xgb_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------
# Threshold optimization (EDGE-BASED, validation only)
# ------------------------------------------------------
best_threshold = 0.5
best_score = -np.inf

baseline = y_val.mean()   # base probability of positive target

for t in np.arange(0.35, 0.65, 0.02):
    preds = (y_val_prob > t).astype(int)
    trade_rate = preds.mean()

    # avoid extreme over/under trading
    if trade_rate < 0.05 or trade_rate > 0.4:
        continue

    if preds.sum() > 0:
        win_rate = y_val[preds == 1].mean()
    else:
        win_rate = 0.0

    edge = win_rate - baseline

    # penalize tiny sample sizes
    score = edge * np.sqrt(preds.sum())

    if score > best_score:
        best_score = score
        best_threshold = t


# ------------------------------------------------------
# Final test predictions
# ------------------------------------------------------
y_test_pred = (y_test_prob > best_threshold).astype(int)

ml_accuracy = accuracy_score(y_test, y_test_pred)
ml_auc = roc_auc_score(y_test, y_test_prob)
ml_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print(f"\n✓ ML Threshold: {best_threshold:.2f}")
print(f"✓ ML Test Accuracy: {ml_accuracy:.4f}")
print(f"✓ ML Test AUC:      {ml_auc:.4f}")
print(f"✓ ML Test F1:       {ml_f1:.4f}")



STEP 1: XGBoost ML MODEL PREDICTIONS (IMPROVED)
X_train: (791543, 14)
X_test:  (80848, 14)
y_train: (791543,)
y_test:  (80848,)

Training XGBoost model...

✓ ML Threshold: 0.35
✓ ML Test Accuracy: 0.6588
✓ ML Test AUC:      0.6041
✓ ML Test F1:       0.3402


In [9]:
test_eval = test_with_features.copy()

test_eval["ml_prob"] = y_test_prob[:len(test_eval)]
test_eval["ml_entry"] = (test_eval["ml_prob"] > best_threshold).astype(int)

test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

baseline = y_test[:len(test_eval)].mean()

def evaluate(name, col):
    entries = test_eval[col]
    trade_rate = entries.mean()
    win_rate = y_test[:len(test_eval)][entries == 1].mean() if entries.sum() > 0 else 0
    edge = win_rate - baseline
    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")



--- STRATEGY ONLY ---
Trade rate: 30.47%
Win rate:   26.42%
Edge:       -1.74%

--- ML ONLY ---
Trade rate: 23.55%
Win rate:   37.35%
Edge:       9.19%

--- STRATEGY + ML ---
Trade rate: 4.92%
Win rate:   36.13%
Edge:       7.97%


In [10]:
# ======================================================
# STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)
# ======================================================
print("\n" + "="*80)
print("STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)")
print("="*80)

# Base evaluation frame (must already be aligned)
test_eval = test_with_features.copy()

# ------------------------------------------------------
# Entries
# ------------------------------------------------------
test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)
test_eval["ml_entry"] = (y_test_prob > best_threshold).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

# Align target
y_target = y_test.values[:len(test_eval)]
baseline = y_target.mean()

# ------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------
def evaluate(name, entry_col):
    entries = test_eval[entry_col]
    trade_rate = entries.mean()
    
    if entries.sum() > 0:
        win_rate = y_target[entries == 1].mean()
    else:
        win_rate = 0.0

    edge = win_rate - baseline

    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

# ------------------------------------------------------
# Results
# ------------------------------------------------------
evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")



STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)

--- STRATEGY ONLY ---
Trade rate: 30.47%
Win rate:   26.42%
Edge:       -1.74%

--- ML ONLY ---
Trade rate: 23.55%
Win rate:   37.35%
Edge:       9.19%

--- STRATEGY + ML ---
Trade rate: 4.92%
Win rate:   36.13%
Edge:       7.97%


In [11]:
# Strategy entries
strategy_entries = (test_aligned["strategy_signal"] == 1).astype(int)
strategy_entries = strategy_entries[:len(y_test_target)]

# ML entries
ml_entries = (y_test_prob_ml > best_threshold).astype(int)
ml_entries = ml_entries[:len(y_test_target)]

# Combined entries
combined_entries = (strategy_entries == 1) & (ml_entries == 1)

# Stats
baseline = y_test_target.mean()

combined_trade_rate = combined_entries.mean()
combined_winrate = y_test_target[combined_entries].mean() if combined_entries.sum() > 0 else 0
combined_edge = combined_winrate - baseline

print("\n--- STRATEGY + ML COMBINED ---")
print(f"Trade rate: {combined_trade_rate:.2%}")
print(f"Win rate:   {combined_winrate:.2%}")
print(f"Edge:       {combined_edge:.2%}")


NameError: name 'test_aligned' is not defined

In [12]:
# ======================================================
# STEP 3: COMBINED ML + STRATEGY PREDICTIONS
# ======================================================
print("\n" + "="*80)
print("STEP 3: COMBINED ML + STRATEGY ENSEMBLE")
print("="*80)

# Align predictions to common test set
ml_preds = y_test_pred_ml
ml_probs = y_test_prob_ml
strategy_preds = y_test_pred_strategy_buy
y_test_common = y_test_ml.values

# Trim to common length
min_len = min(len(ml_preds), len(strategy_preds))
ml_preds = ml_preds[:min_len]
ml_probs = ml_probs[:min_len]
strategy_preds = strategy_preds[:min_len]
y_test_common = y_test_common[:min_len]

print(f"Common test length: {min_len}")
print(f"ML predictions: {len(ml_preds)}")
print(f"Strategy predictions: {len(strategy_preds)}")

# ======================================================
# ENSEMBLE METHODS
# ======================================================

# Method 1: Majority Voting (simple average)
ensemble_prob_voting = (ml_probs + strategy_preds) / 2.0
ensemble_pred_voting = (ensemble_prob_voting > 0.5).astype(int)

# Method 2: Weighted Voting (70% ML, 30% Strategy)
ensemble_prob_weighted = (0.7 * ml_probs) + (0.3 * strategy_preds)
ensemble_pred_weighted = (ensemble_prob_weighted > 0.5).astype(int)

# Method 3: Agreement-Based (only predict if both agree)
ensemble_pred_agreement = ((ml_preds == 1) & (strategy_preds == 1)).astype(int)

# ======================================================
# EVALUATION
# ======================================================
voting_accuracy = accuracy_score(y_test_common, ensemble_pred_voting)
voting_f1 = f1_score(y_test_common, ensemble_pred_voting, zero_division=0)
voting_auc = roc_auc_score(y_test_common, ensemble_prob_voting)

weighted_accuracy = accuracy_score(y_test_common, ensemble_pred_weighted)
weighted_f1 = f1_score(y_test_common, ensemble_pred_weighted, zero_division=0)
weighted_auc = roc_auc_score(y_test_common, ensemble_prob_weighted)

agreement_accuracy = accuracy_score(y_test_common, ensemble_pred_agreement)
agreement_f1 = f1_score(y_test_common, ensemble_pred_agreement, zero_division=0)

print("\n" + "="*80)
print("COMPARISON: ML vs Strategy vs Combined")
print("="*80)

# Calculate metrics on common test set
ml_accuracy_aligned = accuracy_score(y_test_common, ml_preds)
ml_auc_aligned = roc_auc_score(y_test_common, ml_probs)
ml_f1_aligned = f1_score(y_test_common, ml_preds, zero_division=0)

strategy_accuracy_aligned = accuracy_score(y_test_common, strategy_preds)
strategy_auc_aligned = roc_auc_score(y_test_common, strategy_preds)
strategy_f1_aligned = f1_score(y_test_common, strategy_preds, zero_division=0)

print(f"\n{'Approach':<25} {'Accuracy':<12} {'AUC':<12} {'F1':<12}")
print("-" * 61)
print(f"{'ML (XGBoost)':<25} {ml_accuracy_aligned:<12.4f} {ml_auc_aligned:<12.4f} {ml_f1_aligned:<12.4f}")
print(f"{'Strategy (Technical)':<25} {strategy_accuracy_aligned:<12.4f} {strategy_auc_aligned:<12.4f} {strategy_f1_aligned:<12.4f}")
print("-" * 61)
print(f"{'Combined (Voting 50/50)':<25} {voting_accuracy:<12.4f} {voting_auc:<12.4f} {voting_f1:<12.4f}")
print(f"{'Combined (Weighted 70/30)':<25} {weighted_accuracy:<12.4f} {weighted_auc:<12.4f} {weighted_f1:<12.4f}")
print(f"{'Combined (Agreement)':<25} {agreement_accuracy:<12.4f} {'N/A':<12} {agreement_f1:<12.4f}")
print("="*80)

# Best approach
approaches = {
    'ML': ml_accuracy_aligned,
    'Strategy': strategy_accuracy_aligned,
    'Combined (Voting)': voting_accuracy,
    'Combined (Weighted)': weighted_accuracy,
    'Combined (Agreement)': agreement_accuracy
}
best_approach = max(approaches, key=approaches.get)
best_accuracy = approaches[best_approach]

print(f"\n🏆 Best Approach: {best_approach} with accuracy {best_accuracy:.4f}")
print(f"   Improvement over ML: {(best_accuracy - ml_accuracy_aligned)*100:.2f}%")
print(f"   Improvement over Strategy: {(best_accuracy - strategy_accuracy_aligned)*100:.2f}%")


STEP 3: COMBINED ML + STRATEGY ENSEMBLE


NameError: name 'y_test_pred_ml' is not defined

In [97]:
# ======================================================
# STEP 4: DETAILED ANALYSIS
# ======================================================
print("\n" + "="*80)
print("DETAILED ANALYSIS - BEST COMBINED APPROACH")
print("="*80)

# Use weighted ensemble as it's most balanced
print("\nWeighted Ensemble (70% ML + 30% Strategy):")
print("\nClassification Report:")
print(classification_report(y_test_common, ensemble_pred_weighted))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_common, ensemble_pred_weighted))

# Signal distribution
print("\nSignal Distribution:")
print(f"ML Buy Signals:        {ml_preds.sum()} / {len(ml_preds)}")
print(f"Strategy Buy Signals:  {strategy_preds.sum()} / {len(strategy_preds)}")
print(f"Combined Buy Signals:  {ensemble_pred_weighted.sum()} / {len(ensemble_pred_weighted)}")
print(f"Actual Up Moves:       {y_test_common.sum()} / {len(y_test_common)}")

# Agreement analysis
agreement = (ml_preds == strategy_preds).astype(int)
print(f"\nAgreement Rate: {agreement.sum() / len(agreement) * 100:.2f}%")

# Analyze disagreements
disagreement_idx = np.where(ml_preds != strategy_preds)[0]
if len(disagreement_idx) > 0:
    disagreement_correct = y_test_common[disagreement_idx]
    ml_correct_on_disagreements = (ml_preds[disagreement_idx] == disagreement_correct).sum()
    strategy_correct_on_disagreements = (strategy_preds[disagreement_idx] == disagreement_correct).sum()
    
    print(f"\nOn Disagreements ({len(disagreement_idx)} cases):")
    print(f"  ML correct:       {ml_correct_on_disagreements}")
    print(f"  Strategy correct: {strategy_correct_on_disagreements}")


DETAILED ANALYSIS - BEST COMBINED APPROACH

Weighted Ensemble (70% ML + 30% Strategy):

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.68      0.57     40521
           1       0.47      0.29      0.36     40327

    accuracy                           0.49     80848
   macro avg       0.48      0.49      0.47     80848
weighted avg       0.48      0.49      0.47     80848


Confusion Matrix:
[[27589 12932]
 [28644 11683]]

Signal Distribution:
ML Buy Signals:        79127 / 80848
Strategy Buy Signals:  24615 / 80848
Combined Buy Signals:  24615 / 80848
Actual Up Moves:       40327 / 80848

Agreement Rate: 30.00%

On Disagreements (56590 cases):
  ML correct:       28923
  Strategy correct: 27667


## Multi-Ticker Combined Analysis

Apply combined approach to all tickers and compare results.

In [98]:
multi_ticker_results = []

print("\n" + "="*80)
print("MULTI-TICKER ML-ONLY EDGE ANALYSIS")
print("="*80)

for ticker in DEFAULT_TICKERS:
    print(f"\n{ticker}...", end=" ")

    try:
        # Load & clean
        raw_data = load_kaggle_data(ticker)
        cleaned_data = clean_ohlcv_data(raw_data)
        train_data, test_data = split_data_by_date(cleaned_data)

        # Strategy FIRST (alignment)
        train_with_signals = add_scalping_signals(train_data)
        test_with_signals  = add_scalping_signals(test_data)

        # Feature engineering
        train_df = add_basic_features(train_with_signals)
        test_df  = add_basic_features(test_with_signals)

        if len(train_df) < 1000 or len(test_df) < 500:
            print("SKIPPED (insufficient data)")
            continue

        # Features / target
        feature_cols = [
            c for c in train_df.columns
            if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
        ]

        X_train = train_df[feature_cols]
        y_train = train_df["target"]

        X_test = test_df[feature_cols]
        y_test = test_df["target"]

        # Train ML (robust config)
        model = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.7,
            colsample_bytree=0.7,
            min_child_weight=20,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            verbosity=0
        )

        model.fit(X_train, y_train)

        # ML probabilities
        ml_prob = model.predict_proba(X_test)[:, 1]

        # Threshold from earlier logic
        threshold = best_threshold

        ml_entry = (ml_prob > threshold).astype(int)

        baseline = y_test.mean()
        trade_rate = ml_entry.mean()

        if ml_entry.sum() > 0:
            win_rate = y_test[ml_entry == 1].mean()
        else:
            win_rate = 0.0

        edge = win_rate - baseline
        auc = roc_auc_score(y_test, ml_prob)

        multi_ticker_results.append({
            "ticker": ticker,
            "trade_rate": trade_rate,
            "win_rate": win_rate,
            "edge": edge,
            "auc": auc
        })

        print(f"✓ Edge:{edge:+.2%} | AUC:{auc:.3f}")

    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")



MULTI-TICKER ML-ONLY EDGE ANALYSIS

NIFTY BANK... 2025-12-24 16:13:00 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-24 16:13:01 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-24 16:13:01 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-24 16:13:01 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-24 16:13:01 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-24 16:13:01 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-24 16:13:01 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-24 16:13:01 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-0

In [8]:
df = pd.DataFrame(multi_ticker_results)
print(df.sort_values("edge", ascending=False))

print("\nAVERAGES")
print(f"Avg Edge: {df['edge'].mean():+.2%}")
print(f"Avg AUC:  {df['auc'].mean():.3f}")


NameError: name 'multi_ticker_results' is not defined

## Key Findings

Summary of the combined ML + Strategy approach compared to individual techniques.

In [13]:
# =====================================================
# BACKTEST CONFIG (MAX-RETURN READY)
# =====================================================

INITIAL_CAPITAL = 1_000_000
RISK_FREE_RATE = 0.0

HORIZON =24

# -------------------------------
# ML ENTRY (TAIL ONLY)
# -------------------------------
ENTRY_Q = 0.985      # trade only top 5% signals

# -------------------------------
# EXECUTION CONTROLS
# -------------------------------
STOP_LOSS = 0.004
COST_PER_TRADE = 0.00015

# Position sizing (convex)
SIZE_EXPONENT = 3
MAX_POSITION = 1.0



# Trade management
COOLDOWN = HORIZON//2


In [14]:
ticker = "NIFTY BANK"
print(f"\nBacktesting: {ticker}")

# --------------------------------------------------
# Load & clean
# --------------------------------------------------
raw = load_kaggle_data(ticker)
cleaned = clean_ohlcv_data(raw)
train_data, test_data = split_data_by_date(cleaned)

# --------------------------------------------------
# CONTINUOUS FEATURE PIPELINE (CRITICAL FIX)
# --------------------------------------------------
# Combine train + test to preserve rolling context
full_data = pd.concat([train_data, test_data], axis=0)

# Apply features on full history
full_with_signals = add_scalping_signals(full_data)
full_features = add_basic_features(full_with_signals)

# Slice back test portion ONLY
test_df = full_features.loc[test_data.index]

# --------------------------------------------------
# ML inputs
# --------------------------------------------------
feature_cols = [
    c for c in test_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_test = test_df[feature_cols]
y_test = test_df["target"]
prices = test_df["Close"].values



Backtesting: NIFTY BANK
2025-12-24 18:32:04 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-24 18:32:05 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-24 18:32:05 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-24 18:32:05 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-24 18:32:05 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-24 18:32:05 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-24 18:32:05 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-24 18:32:05 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-1

In [15]:
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# TRAIN FEATURES (DEFINE FEATURE SPACE HERE)
# --------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
train_df = add_basic_features(train_with_signals)

feature_cols = [
    c for c in train_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_train = train_df[feature_cols]
y_train = train_df["target"]

# --------------------------------------------------
# SCALE (FIT ON TRAIN ONLY)
# --------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------------
# MODEL
# --------------------------------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=20,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

model.fit(X_train_scaled, y_train)

# --------------------------------------------------
# ML PROBABILITIES
# --------------------------------------------------
ml_prob = model.predict_proba(X_test_scaled)[:, 1]
print("Test AUC:", roc_auc_score(y_test, ml_prob))


Test AUC: 0.6049132125626437


In [16]:
def position_size(prob, threshold):
    """
    Convert ML probability into position size [0, 1]
    """
    size = (prob - threshold) / (1 - threshold)
    return np.clip(size, 0, 1)


In [17]:
capital = INITIAL_CAPITAL
equity_curve = []
trades = []

ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)
STOP_LOSS = 0.002          # -0.20%
TAKE_PROFIT = 0.01         # +1.0%
COOLDOWN = HORIZON

i = 0
n = len(prices)

while i < n - HORIZON:

    prob = ml_prob[i]

    # --------------------------------------------------
    # SKIP LOW-CONFIDENCE BARS (FAST)
    # --------------------------------------------------
    if prob <= ENTRY_THRESHOLD:
        equity_curve.append(capital)
        i += 1
        continue

    # --------------------------------------------------
    # POSITION SIZING
    # --------------------------------------------------
    edge_strength = (prob - ENTRY_THRESHOLD) / (1 - ENTRY_THRESHOLD)
    size = np.clip(edge_strength ** SIZE_EXPONENT, 0.25, 1.0)

    vol = test_df["volatility_10"].iloc[i]
    risk_adj = np.clip(0.01 / vol, 0.5, 2.0)

    position_value = capital * size * risk_adj

    entry_price = prices[i]

    # --------------------------------------------------
    # EXIT LOGIC (BAR-BY-BAR)
    # --------------------------------------------------
    exit_price = prices[i + HORIZON]
    exit_idx = i + HORIZON

    for j in range(1, HORIZON + 1):
        price = prices[i + j]

        # STOP LOSS
        if price <= entry_price * (1 - STOP_LOSS):
            exit_price = entry_price * (1 - STOP_LOSS)
            exit_idx = i + j
            break

        # TAKE PROFIT
        if price >= entry_price * (1 + TAKE_PROFIT):
            exit_price = entry_price * (1 + TAKE_PROFIT)
            exit_idx = i + j
            break

    # --------------------------------------------------
    # PnL CALCULATION (ONCE PER TRADE)
    # --------------------------------------------------
    ret = (exit_price - entry_price) / entry_price
    net_ret = ret - COST_PER_TRADE

    pnl = position_value * net_ret
    capital += pnl

    trades.append({
        "entry_idx": i,
        "exit_idx": exit_idx,
        "prob": prob,
        "size": size,
        "return": net_ret,
        "pnl": pnl,
        "capital": capital
    })

    equity_curve.append(capital)

    # --------------------------------------------------
    # COOLDOWN
    # --------------------------------------------------
    i += COOLDOWN


In [18]:
equity = pd.Series(equity_curve)
returns = equity.pct_change().dropna()

total_return = (equity.iloc[-1] / equity.iloc[0]) - 1
max_dd = ((equity / equity.cummax()) - 1).min()

sharpe = (
    returns.mean() / returns.std()
    if returns.std() > 0 else 0
) * np.sqrt(252 * 6.5 * 60)   # intraday annualization

profit_factor = (
    sum(t["pnl"] for t in trades if t["pnl"] > 0) /
    abs(sum(t["pnl"] for t in trades if t["pnl"] < 0))
    if any(t["pnl"] < 0 for t in trades) else np.inf
)

print("\n================ BACKTEST SUMMARY ================")
print(f"Final Capital:     ₹{capital:,.0f}")
print(f"Total Return:      {total_return*100:.2f}%")
print(f"Max Drawdown:      {max_dd*100:.2f}%")
print(f"Sharpe Ratio:      {sharpe:.2f}")
print(f"Profit Factor:    {profit_factor:.2f}")
print(f"Total Trades:     {len(trades)}")



================ BACKTEST SUMMARY ================
Final Capital:     ₹1,050,653
Total Return:      5.07%
Max Drawdown:      -0.89%
Sharpe Ratio:      2.31
Profit Factor:    1.34
Total Trades:     299


In [19]:
trade_returns = [t["return"] for t in trades]

print("\nTrade Stats")
print(f"Win rate: {np.mean(np.array(trade_returns) > 0)*100:.2f}%")
print(f"Avg win:  {np.mean([r for r in trade_returns if r > 0])*100:.2f}%")
print(f"Avg loss: {np.mean([r for r in trade_returns if r < 0])*100:.2f}%")



Trade Stats
Win rate: 46.15%
Avg win:  0.28%
Avg loss: -0.18%


In [20]:
TICKERS = [
    "NIFTY BANK",
    "NIFTY FIN SERVICE",
    "NIFTY CONSUMPTION",
    "NIFTY INDIA MFG"
]

PORTFOLIO_CAPITAL = 1_000_000
CAPITAL_PER_TICKER = PORTFOLIO_CAPITAL / len(TICKERS)

portfolio_results = []
portfolio_equity_curves = []


In [21]:
for ticker in TICKERS:
    print(f"\nBacktesting {ticker}...")

    # -----------------------------
    # Load & prepare data
    # -----------------------------
    raw = load_kaggle_data(ticker)
    cleaned = clean_ohlcv_data(raw)
    train_data, test_data = split_data_by_date(cleaned)

    test_with_signals = add_scalping_signals(test_data)
    test_df = add_basic_features(test_with_signals)

    feature_cols = [
        c for c in test_df.columns
        if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
    ]

    X_test = test_df[feature_cols]
    y_test = test_df["target"]
    prices = test_df["Close"].values

    # -----------------------------
    # Train ML (per ticker)
    # -----------------------------
    train_with_signals = add_scalping_signals(train_data)
    train_df = add_basic_features(train_with_signals)

    X_train = train_df[feature_cols]
    y_train = train_df["target"]

    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=20,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        verbosity=0
    )

    model.fit(X_train, y_train)
    ml_prob = model.predict_proba(X_test)[:, 1]

    # -----------------------------
    # Backtest (same logic)
    # -----------------------------
    ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)

    capital = CAPITAL_PER_TICKER
    equity_curve = []
    trades = []

    i = 0
    while i < len(prices) - HORIZON:
        prob = ml_prob[i]

        if prob > ENTRY_THRESHOLD:
            edge = (prob - ENTRY_THRESHOLD) / (1 - ENTRY_THRESHOLD)
            size = np.clip(edge ** SIZE_EXPONENT, 0, 1)
            size = max(size, 0.25)

            position_value = capital * size
            entry_price = prices[i]

            window = prices[i+1:i+HORIZON+1]
            min_price = window.min()

            if (min_price - entry_price) / entry_price <= -STOP_LOSS:
                exit_price = entry_price * (1 - STOP_LOSS)
            else:
                exit_price = prices[i + HORIZON]

            ret = (exit_price - entry_price) / entry_price
            net_ret = ret - COST_PER_TRADE
            pnl = position_value * net_ret
            capital += pnl

            trades.append(pnl)
            equity_curve.append(capital)

            i += COOLDOWN
        else:
            equity_curve.append(capital)
            i += 1

    # -----------------------------
    # Metrics
    # -----------------------------
    equity = np.array(equity_curve)
    returns = np.diff(equity) / equity[:-1]

    total_return = (capital / CAPITAL_PER_TICKER - 1) * 100
    max_dd = ((equity / np.maximum.accumulate(equity)) - 1).min() * 100


    profit_factor = (
        sum(p for p in trades if p > 0) /
        abs(sum(p for p in trades if p < 0))
        if any(p < 0 for p in trades) else np.inf
    )

    portfolio_results.append({
        "Ticker": ticker,
        "Return (%)": total_return,
        "Max DD (%)": max_dd,
        "Trades": len(trades),
        "Profit Factor": profit_factor
    })

    portfolio_equity_curves.append(equity)

    print(f"✓ Return: {total_return:.2f}% | DD: {max_dd:.2f}% | PF: {profit_factor:.2f}")



Backtesting NIFTY BANK...
2025-12-24 18:32:40 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-24 18:32:41 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-24 18:32:41 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-24 18:32:41 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-24 18:32:41 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-24 18:32:41 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-24 18:32:41 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-24 18:32:41 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04

KeyboardInterrupt: 

In [286]:
results_df = pd.DataFrame(portfolio_results)
print("\n=== PER-TICKER RESULTS ===")
display(results_df)

# -----------------------------
# Portfolio aggregation
# -----------------------------
min_len = min(len(eq) for eq in portfolio_equity_curves)
portfolio_equity = sum(eq[:min_len] for eq in portfolio_equity_curves)

portfolio_return = (portfolio_equity[-1] / PORTFOLIO_CAPITAL - 1) * 100
portfolio_dd = ((portfolio_equity / np.maximum.accumulate(portfolio_equity)) - 1).min() * 100

print("\n=== PORTFOLIO SUMMARY ===")
print(f"Final Capital: ₹{portfolio_equity[-1]:,.0f}")
print(f"Total Return:  {portfolio_return:.2f}%")
print(f"Max Drawdown:  {portfolio_dd:.2f}%")



=== PER-TICKER RESULTS ===


,Ticker,Return (%),Max DD (%),Trades,Profit Factor
0,NIFTY BANK,3.515105,-0.721874,631,1.229770
1,NIFTY FIN SERVICE,4.349549,-1.293197,594,1.312716
2,NIFTY CONSUMPTION,3.277902,-1.049602,456,1.305934
3,NIFTY INDIA MFG,-0.253204,-1.720055,777,0.986451



=== PORTFOLIO SUMMARY ===
Final Capital: ₹1,021,159
Total Return:  2.12%
Max Drawdown:  -0.59%
